In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load Excel data
def load_excel(file_path):
    df = pd.read_csv(file_path)
    summary = "Excel Data:\n"
    for idx, row in df.iterrows():
        # Handle missing or 'NA' values
        breaking_capacity = row['Rated Breaking Capacity (A)'] if pd.notna(row['Rated Breaking Capacity (A)']) and row['Rated Breaking Capacity (A)'] != 'NA' else 'Unknown'
        application = row['Application'] if pd.notna(row['Application']) and row['Application'] != 'NA' else 'Unspecified'
        summary += f"{row['DESCRIPTION']}: {breaking_capacity} breaking capacity for {application}\n"
    return summary

# Load Phi-3.5 model
model_name = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Chatbot function
def chat_with_phi(excel_data, user_query):
    prompt = f"{excel_data}\n\nQuery: {user_query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Answer:")[1].strip()


c:\Users\ranga\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:44<00:00, 22.41s/it]


In [ ]:
if __name__ == "__main__":
    file_path = rf"C:\Users\ranga\OneDrive\Desktop\BMW-Tech-Works\Task 1\Parts_processed.csv"
    excel_data = load_excel(file_path)
    # print("Excel Data Loaded:\n", excel_data)
    
    while True:
        query = input("Ask a question (or type 'exit' to quit): ")
        if query.lower() == "exit":
            break
        response = chat_with_phi(excel_data, query)
        print("Phi-3.5 Response:", response)

c:\Users\ranga\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [6]:
!pip install gradio

  Obtaining dependency information for gradio from https://files.pythonhosted.org/packages/93/e9/dfc030f623a8c5efce02b853cfe1b9a47dd1365cc028926800b5757308f1/gradio-5.21.0-py3-none-any.whl.metadata
  Obtaining dependency information for aiofiles<24.0,>=22.0 from https://files.pythonhosted.org/packages/c5/19/5af6804c4cc0fed83f47bff6e413a98a36618e7d40185cd36e69737f3b0e/aiofiles-23.2.1-py3-none-any.whl.metadata
  Using cached aiofiles-23.2.1-py3-none-any.whl.metadata (9.7 kB)
  Obtaining dependency information for anyio<5.0,>=3.0 from https://files.pythonhosted.org/packages/46/eb/e7f063ad1fec6b3178a3cd82d1a3c4de82cccf283fc42746168188e1cdd5/anyio-4.8.0-py3-none-any.whl.metadata
  Using cached anyio-4.8.0-py3-none-any.whl.metadata (4.6 kB)
  Obtaining dependency information for fastapi<1.0,>=0.115.2 from https://files.pythonhosted.org/packages/b3/5d/4d8bbb94f0dbc22732350c06965e40740f4a92ca560e90bb566f4f73af41/fastapi-0.115.11-py3-none-any.whl.metadata
  Using cached fastapi-0.115.11-py3-non


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr

# Load Excel data
def load_excel(file_path):
    df = pd.read_csv(file_path)
    summary = "Excel Data:\n"
    for idx, row in df.iterrows():
        breaking_capacity = row['Rated Breaking Capacity (A)'] if pd.notna(row['Rated Breaking Capacity (A)']) and row['Rated Breaking Capacity (A)'] != 'NA' else 'Unknown'
        application = row['Application'] if pd.notna(row['Application']) and row['Application'] != 'NA' else 'Unspecified'
        summary += f"{row['DESCRIPTION']}: {breaking_capacity} breaking capacity for {application}\n"
    return summary

# Load and quantize Phi-3.5 model
model_name = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)  # Use FP16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Chatbot function
def chat_with_phi(excel_data, user_query):
    prompt = f"{excel_data}\n\nQuery: {user_query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Answer:")[1].strip()

def chatbot_interface(query):
    global excel_data  # Access the pre-loaded excel_data
    if not query.strip():  # Check for empty input
        return "Please enter a question."
    response = chat_with_phi(excel_data, query)
    return response

Loading checkpoint shards: 100%|██████████| 2/2 [01:52<00:00, 56.16s/it]


In [ ]:
if __name__ == "__main__":
    file_path = r"C:\Users\ranga\OneDrive\Desktop\BMW-Tech-Works\Task 1\Parts_processed.csv"
    excel_data = load_excel(file_path)  # Load data once at startup
    print("Excel Data Loaded (length:", len(excel_data), "characters)")

    # Create Gradio interface
    interface = gr.Interface(
        fn=chatbot_interface,
        inputs=gr.Textbox(label="Ask a question", placeholder="Type your question here (e.g., 'Which parts have 1500A breaking capacity?')"),
        outputs=gr.Textbox(label="Phi-3.5 Response"),
        title="BMW Parts Chatbot",
        description="Ask questions about parts from Parts_processed.csv. Type 'exit' does not work here; use the interface to interact.",
        theme="default"
    )

    # Launch the interface
    interface.launch()

In [9]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr

# Load CSV data
def load_csv(file_path):
    df = pd.read_csv(file_path)
    summary = "Excel Data:\n"
    for idx, row in df.iterrows():
        breaking_capacity = row['Rated Breaking Capacity (A)'] if pd.notna(row['Rated Breaking Capacity (A)']) and row['Rated Breaking Capacity (A)'] != 'NA' else 'Unknown'
        application = row['Application'] if pd.notna(row['Application']) and row['Application'] != 'NA' else 'Unspecified'
        summary += f"{row['DESCRIPTION']}: {breaking_capacity} breaking capacity for {application}\n"
        if idx > 100:  # Limit data size for faster processing
            summary += "...\n(Data truncated for faster response)"
            break
    return summary

# Load and quantize Phi-3.5 model
model_name = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Chatbot function
def chat_with_phi(excel_data, user_query):
    prompt = f"{excel_data}\n\nQuery: {user_query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Answer:")[-1].strip()

def chatbot_interface(query):
    global excel_data
    if not query.strip():
        return "Please enter a question."
    response = chat_with_phi(excel_data, query)
    return response

if __name__ == "__main__":
    file_path = rf"C:\Users\ranga\OneDrive\Desktop\BMW-Tech-Works\Task 1\Parts_processed.csv"
    excel_data = load_csv(file_path)
    print("Excel Data Loaded (length:", len(excel_data), "characters)")

    interface = gr.Interface(
        fn=chatbot_interface,
        inputs=gr.Textbox(label="Ask a question", placeholder="Type your question here"),
        outputs=gr.Textbox(label="Phi-3.5 Response"),
        title="BMW Parts Chatbot",
        description="Ask questions about parts from Parts_processed.csv.",
        theme="default"
    )

    interface.launch()


Loading checkpoint shards: 100%|██████████| 2/2 [00:37<00:00, 18.60s/it]

KeyboardInterrupt



In [12]:
!pip install faiss-cpu

  Obtaining dependency information for faiss-cpu from https://files.pythonhosted.org/packages/2c/2d/d2a4171a9cca9a7c04cd9d6f9441a37f1e0558724b90bf7fc7db08553601/faiss_cpu-1.10.0-cp311-cp311-win_amd64.whl.metadata
  Using cached faiss_cpu-1.10.0-cp311-cp311-win_amd64.whl.metadata (4.5 kB)
Using cached faiss_cpu-1.10.0-cp311-cp311-win_amd64.whl (13.7 MB)



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Load the CSV file
data = pd.read_csv(rf"C:\Users\ranga\OneDrive\Desktop\BMW-Tech-Works\Task 1\Parts_processed.csv")  

# Convert each row to a text representation.
# This example concatenates all columns; you may want to tailor this per your data.
documents = data.astype(str).apply(lambda row: " | ".join(row.values), axis=1).tolist()

# 2. Generate embeddings for each document
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # a lightweight open-source model
doc_embeddings = embedder.encode(documents, show_progress_bar=True)
doc_embeddings = np.array(doc_embeddings).astype("float32")

# 3. Create a FAISS index and add the embeddings
embedding_dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(doc_embeddings)
print(f"Indexed {index.ntotal} documents.")

# 4. Load an open source language model for generating answers
# model_name = "EleutherAI/gpt-neo-125M"  # smaller model for demo; you can choose a larger one if resources allow
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)
model_name = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).eval()
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def get_relevant_context(query, k=3):
    # Encode the query using the same embedder
    query_embedding = embedder.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")
    # Retrieve the nearest documents
    distances, indices = index.search(query_embedding, k)
    retrieved_docs = [documents[idx] for idx in indices[0]]
    return "\n".join(retrieved_docs)

def generate_answer(query):
    # Retrieve context
    context = get_relevant_context(query, k=3)
    # Build a prompt that includes context and the user question
    prompt = (
        f"Based on the following data from the CSV file:\n{context}\n\n"
        f"Answer the following question accurately:\n{query}\n"
    )
    # Generate answer using the LLM
    generated = generator(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
    answer = generated[0]['generated_text'][len(prompt):].strip()
    return answer

# Example query
if __name__ == "__main__":
    user_query = "give first 5 ID's?"  # replace with a real query
    answer = generate_answer(user_query)
    print("Answer:", answer)


Batches: 100%|██████████| 31/31 [00:12<00:00,  2.53it/s]


Indexed 992 documents.


Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.28s/it]
